In [5]:
!brew services start ollama

==> Successfully started `ollama` (label: homebrew.mxcl.ollama)


In [7]:
!ollama pull all-minilm

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 797b70c4edf8: 100% ▕██████████████████▏  45 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling 85011998c600: 100% ▕██████████████████▏   16 B                         
pulling 548455b72658: 100% ▕██████████████████▏  407 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
import sys
import ollama
import numpy as np

# 1. YOUR SPECIALIZED KNOWLEDGE BASE (STILL INTACT)
KNOWLEDGE_BASE = [
    "In the 2026 World Cup matches on June 22, France defeated Iraq with a definitive final score of 3-0 in their Group I matchup.",
    "In a Group I match on June 22, Norway played Senegal. The match saw Erling Haaland score two spectacular goals, bringing the score to 3-2 in favor of Norway late in the 90th minute.",
    "During the Group J match on June 22, Argentina defeated Austria 2-0. Lionel Messi scored both goals, making history by becoming the all-time leading goalscorer in World Cup history and securing Argentina's advance to the Round of 32.",
    "On June 21, Spain dominated Saudi Arabia in their Group H matchup, finishing with a final score of 4-0.",
    "On June 21, the Group G match between Belgium and Iran ended in a physical, defensive 0-0 scoreless draw at full-time.",
    "On June 21, Uruguay tied with Cabo Verde in a high-scoring Group H match that ended in a 2-2 draw.",
    "On June 21, Egypt secured a decisive victory against New Zealand in Group G, winning the match with a final score of 3-1.",
    "On June 20, the Group E match between Ecuador and tournament underdogs Curaçao concluded in a tight 0-0 draw at full-time.",
    "On June 20, Japan put on an offensive masterclass against Tunisia in Group F, winning the match with a clear final score of 4-0."
]

def get_embedding(text: str, model: str = "all-minilm") -> list:
    try:
        response = ollama.embeddings(model=model, prompt=text)
        return response['embedding']
    except Exception as e:
        print(f"\n[Error generating embedding]: Detail: {e}")
        sys.exit(1)

def cosine_similarity(v1, v2) -> float:
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("--> Indexing specialized Knowledge Base...")
DOCUMENT_EMBEDDINGS = [get_embedding(doc) for doc in KNOWLEDGE_BASE]
print("--> Hybrid system initialized.\n")

def retrieve_relevant_context(query: str, similarity_threshold: float = 0.4) -> str:
    """
    Pulls context ONLY if it closely matches the query. 
    If the best match falls below the threshold, we return an empty string 
    so we don't pollute Gemma's prompt with irrelevant facts.
    """
    query_vector = get_embedding(query)
    best_score = -1
    best_doc = ""

    for idx, doc_vector in enumerate(DOCUMENT_EMBEDDINGS):
        score = cosine_similarity(query_vector, doc_vector)
        if score > best_score:
            best_score = score
            best_doc = KNOWLEDGE_BASE[idx]
    
    # Only supply context if it's genuinely relevant to the conversation
    if best_score >= similarity_threshold:
        return f"[PROPRIETARY REFERENCE DATA]: {best_doc}"
    return ""

def run_chat_session():
    # Maintain conversational context
    session_history = []
    
    print("=====================================================================")
    print("      Gemma-2:2b Hybrid Assistant (Type 'exit' to quit)")
    print("=====================================================================")
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
            if not user_input or user_input.lower() == 'exit':
                print("Ja ne!")
                break
            
            # 1. Dynamically pull context if relevant
            context = retrieve_relevant_context(user_input, similarity_threshold=0.35)
            
            # 2. TWEEKED PROMPT ARCHITECTURE: The Hybrid Directive
            system_instruction = (
                "You are an intelligent, versatile AI assistant powered by Gemma-2. "
                "You possess broad global knowledge, but you also have access to a small "
                "set of proprietary reference data for specific local facts.\n\n"
                "CRITICAL INSTRUCTIONS:\n"
                "1. If relevant data is present in the 'PROPRIETARY REFERENCE DATA' block below, "
                "you MUST prioritize it to ensure factual accuracy for those specific topics.\n"
                "2. If the reference block is empty or doesn't answer the question, do NOT say "
                "'Information not found'. Instead, use your own pre-trained knowledge base to "
                "answer the user thoroughly and naturally.\n\n"
                f"{context}"
            ).strip()
            
            # Compile payload matching Gemma's template structure
            messages = [{"role": "system", "content": system_instruction}]
            messages.extend(session_history)
            messages.append({"role": "user", "content": user_input})
            
            print("Gemma-2: ", end="", flush=True)
            
            # Slightly raised temperature (0.4) allows natural, creative language generation
            # while keeping it grounded when context is supplied.
            stream = ollama.chat(
                model="gemma2:2b",
                messages=messages,
                stream=True,
                options={"temperature": 0.4} 
            )
            
            full_response_text = ""
            for chunk in stream:
                content = chunk['message']['content']
                print(content, end="", flush=True)
                full_response_text += content
            print()
            
            # Commit exchange to memory
            session_history.append({"role": "user", "content": user_input})
            session_history.append({"role": "assistant", "content": full_response_text})
            
            # Limit memory footprint to last 6 exchanges to prevent performance degradation
            if len(session_history) > 12:
                session_history = session_history[-12:]
                
        except KeyboardInterrupt:
            break

if __name__ == "__main__":
    run_chat_session()

--> Indexing specialized Knowledge Base...
--> Hybrid system initialized.

      Gemma-2:2b Hybrid Assistant (Type 'exit' to quit)



You:  Which national team took championship for World Cup 2022?


Gemma-2: Argentina won the World Cup in 2022. 🏆 🇦🇷  




You:  How's the match of Japan vs Tunisia on June 20th?


Gemma-2: Japan absolutely dominated their match against Tunisia on June 20th, winning with a resounding scoreline of 4-0! 🎉 It was a masterclass performance from the Japanese team.  



In [4]:
!brew services stop ollama

Stopping `ollama`... (might take a while)
==> Successfully stopped `ollama` (label: homebrew.mxcl.ollama)
